### DATA TRANSFORMATION MODULE

### Workflows


1. **Config.yaml**: Este archivo contendrá toda la configuración relacionada con el proyecto. Se utilizará para almacenar todos los parámetros relacionados con el proyecto, así como todas las rutas del proyecto. También se usará para guardar todos los hiperparámetros.
2. **Params.yaml**: Este archivo contendrá todos los hiperparámetros relacionados con el proyecto. Se utilizará para almacenar los hiperparámetros del proyecto y, específicamente, los hiperparámetros del entrenamiento del modelo.
3. **Entidad de configuración (Config entity)**: Este componente definirá la estructura de los archivos de configuración y proporcionará una forma de acceder a los parámetros de configuración de manera estructurada.
4. **Administrador de configuración (Configuration Manager)**: Este componente gestionará la carga y actualización de los archivos de configuración, asegurando que se utilice la configuración correcta en todo el proyecto.
5. **Actualizar componentes**: Ingesta de datos, Transformación de datos, Entrenador del modelo.
6. **Crear pipelines**: Pipeline de entrenamiento, Pipeline de predicción.

In [1]:
import os
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer\\research'

In [2]:
os.chdir('..')
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer'

### Data Transformation

1- Creamos Dataclass
2- Creamos el configuration manager

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [7]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [8]:
%pwd

'c:\\Users\\mecenturion\\Desktop\\MARTIN\\UDEMY\\MLOPS\\TextSummarizer\\textsummarizer'

In [14]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        data_transformation_config = DataTransformationConfig(
            root_dir = Path(config.root_dir),
            data_path = Path(config.data_path),
            tokenizer_name = config.tokenizer_name)
        
        return data_transformation_config
    



In [15]:
import os
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset

In [20]:
# DATA TRANSFORMATION COMPONENT

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):

        model_inputs = self.tokenizer(
            example_batch["dialogue"],
            max_length=1024,
            truncation=True,
            padding=True
        )

        labels = self.tokenizer(
            text_target=example_batch["summary"],
            max_length=128,
            truncation=True,
            padding=True
        )

        model_inputs["labels"] = labels["input_ids"]

        return model_inputs
    
    def convert(self):
        dataset = load_dataset(
            "csv",
            data_files={
                "train": str(self.config.data_path / "train.csv"),
                "validation": str(self.config.data_path / "val.csv"),
                "test": str(self.config.data_path / "test.csv")
            }
        )
        dataset_samsun_pt = dataset.map(self.convert_examples_to_features, batched = True)
        dataset_samsun_pt.save_to_disk(str(self.config.root_dir / "samsum_dataset"))

        

In [21]:
config = ConfigurationManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-04-19 11:40:13,042: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\config\config.yaml' read successfully.]
[2026-04-19 11:40:13,046: INFO: common: 31: YAML file 'C:\Users\mecenturion\Desktop\MARTIN\UDEMY\MLOPS\TextSummarizer\textsummarizer\params.yaml' read successfully.]
[2026-04-19 11:40:13,048: INFO: common: 53: Directory 'artifacts' created successfully.]
[2026-04-19 11:40:13,239: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-04-19 11:40:13,256: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-04-19 11:40:13,386: INFO: _client: 1025: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Re

Generating train split: 14731 examples [00:00, 100254.57 examples/s]
Generating validation split: 818 examples [00:00, 39880.75 examples/s]
Generating test split: 819 examples [00:00, 43106.76 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 819/819 [00:00<00:00, 41257.43 examples/s]
